In [ ]:
import importlib
import sys

# Force reload of helpers module to pick up latest changes
if 'helpers' in sys.modules:
    importlib.reload(sys.modules['helpers'])
    importlib.reload(sys.modules['helpers.database'])
    importlib.reload(sys.modules['helpers.logging_config'])

In [0]:
# ==============================================================================
# CONFIGURATION & HELPER FUNCTIONS
# ==============================================================================
from helpers import create_connection, load_bronze_table, write_gold_table, setup_logger
import time

# Setup logging
logger = setup_logger("load_dimensions")

# Initialize connection
c = create_connection(spark, dbutils)
logger.info("=" * 70)
logger.info("DIMENSION LOAD JOB STARTED")
logger.info("=" * 70)


In [0]:
# ==============================================================================
# BRONZE LAYER: Load all source tables (in-memory DataFrames)
# ==============================================================================

inventory_bronze = load_bronze_table(c, "inventory")
car_bronze = load_bronze_table(c, "car")
inventory_equipment_bronze = load_bronze_table(c, "inventory_equipment")
equipment_bronze = load_bronze_table(c, "equipment")
staff_bronze = load_bronze_table(c, "staff")
city_bronze = load_bronze_table(c, "city")
address_bronze = load_bronze_table(c, "address")
store_bronze = load_bronze_table(c, "store")
country_bronze = load_bronze_table(c, "country")
customer_bronze = load_bronze_table(c, "customer")


In [ ]:
# ==============================================================================
# SILVER LAYER: Clean and prepare data (in-memory transformations)
# ==============================================================================
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lower, when, current_timestamp, lit

logger.info("SILVER: Starting data cleaning")

# Clean inventory: Fix fuel type typos
inventory_silver = inventory_bronze.withColumn(
    "fuel_type",
    when(lower(col("fuel_type")) == "diesle", "Diesel")
     .when(lower(col("fuel_type")) == "petol", "Petrol")
     .otherwise(col("fuel_type"))
)

# Other tables: Pass through (no cleaning needed for now)
car_silver = car_bronze
equipment_silver = equipment_bronze
staff_silver = staff_bronze
city_silver = city_bronze
address_silver = address_bronze
store_silver = store_bronze
country_silver = country_bronze
customer_silver = customer_bronze
inventory_equipment_silver = inventory_equipment_bronze

# Build common denormalized views for gold layer
city_with_country = city_silver.join(country_silver, "country_id", "left")
address_with_location = address_silver.join(city_with_country, "city_id", "left")

logger.info("SILVER: Data cleaning completed")


In [0]:
# ==============================================================================
# GOLD: DIM_DATE
# ==============================================================================
from datetime import date

logger.info("GOLD: Building dim_date")
start_time = time.time()

START_DATE = date(2000, 1, 1)
END_DATE = date(2027, 12, 31)

# Generate date sequence using explode and sequence
dates_df = spark.range(1).select(
    F.explode(
        F.sequence(
            F.lit(START_DATE),
            F.lit(END_DATE)
        )
    ).alias('date')
)

# Build date dimension with all attributes
dim_date = dates_df.select(
    F.col('date'),

    # Date hierarchy
    F.dayofweek('date').alias('day_of_week'),
    F.when(F.dayofweek('date') == 1, 'Monday')
     .when(F.dayofweek('date') == 2, 'Tuesday')
     .when(F.dayofweek('date') == 3, 'Wednesday')
     .when(F.dayofweek('date') == 4, 'Thursday')
     .when(F.dayofweek('date') == 5, 'Friday')
     .when(F.dayofweek('date') == 6, 'Saturday')
     .otherwise('Sunday').alias('day_of_week_name'),

    F.dayofmonth('date').alias('day_of_month'),
    F.weekofyear('date').alias('week_of_year'),
    F.month('date').alias('month'),

    F.when(F.month('date') ==  1, 'January')
     .when(F.month('date') ==  2, 'February')
     .when(F.month('date') ==  3, 'March')
     .when(F.month('date') ==  4, 'April')
     .when(F.month('date') ==  5, 'May')
     .when(F.month('date') ==  6, 'June')
     .when(F.month('date') ==  7, 'July')
     .when(F.month('date') ==  8, 'August')
     .when(F.month('date') ==  9, 'September')
     .when(F.month('date') == 10, 'October')
     .when(F.month('date') == 11, 'November')
     .otherwise('December').alias('month_name'),

    F.quarter('date').alias('quarter'),
    F.year('date').alias('year'),

    # Business flags
    (F.dayofweek('date').isin(6, 7)).alias('is_weekend'),

    # COVID period flags
    (F.col('date') < F.lit(date(2020, 3, 1))).alias('is_pre_covid'),
    (F.col('date').between(F.lit(date(2020, 3, 1)), F.lit(date(2022, 6, 30)))).alias('is_covid'),
    (F.col('date') > F.lit(date(2022, 6, 30))).alias('is_post_covid')
)

# Add date_key (hash of date)
dim_date_final = dim_date.withColumn('date_key', F.xxhash64(F.col('date')))

write_gold_table(dim_date_final, "dim_date")
logger.info(f"GOLD: dim_date completed in {time.time() - start_time:.2f}s")


In [ ]:
# ==============================================================================
# GOLD: DATE DIMENSION COPIES (For BI role-playing dimensions)
# ==============================================================================

logger.info("GOLD: Building date dimension copies for different date roles")
start_time = time.time()

# DIM_SERVICE_DATE - For fact_service.service_date_key
dim_service_date = dim_date_final.withColumnRenamed("date_key", "service_date_key")
write_gold_table(dim_service_date, "dim_service_date")

# DIM_RENTAL_DATE - For fact_rental.rental_date_key
dim_rental_date = dim_date_final.withColumnRenamed("date_key", "rental_date_key")
write_gold_table(dim_rental_date, "dim_rental_date")

# DIM_RETURN_DATE - For fact_rental.return_date_key
dim_return_date = dim_date_final.withColumnRenamed("date_key", "return_date_key")
write_gold_table(dim_return_date, "dim_return_date")

# DIM_PAYMENT_DATE - For fact_rental.payment_date_key
dim_payment_date = dim_date_final.withColumnRenamed("date_key", "payment_date_key")
write_gold_table(dim_payment_date, "dim_payment_date")

# DIM_PAYMENT_DEADLINE_DATE - For fact_rental.payment_deadline_date_key
dim_payment_deadline_date = dim_date_final.withColumnRenamed("date_key", "payment_deadline_date_key")
write_gold_table(dim_payment_deadline_date, "dim_payment_deadline_date")

logger.info(f"GOLD: All date dimension copies completed in {time.time() - start_time:.2f}s")


In [0]:
# ==============================================================================
# GOLD: DIM_STAFF + BRIDGE_STAFF_HIERARCHY
# ==============================================================================
from pyspark.sql.functions import xxhash64
from pyspark.sql import Window

logger.info("GOLD: Building dim_staff and staff hierarchy")
start_time = time.time()

# Build staff hierarchy with surrogate keys
staff_hierarchy = staff_silver.select(
    F.col("staff_id"),
    F.col("manager_id"),
    xxhash64(F.col("staff_id")).alias("staff_key"),
    xxhash64(F.col("manager_id")).alias("staff_manager_key"),
    F.col("first_name").alias("staff_first_name"),
    F.col("last_name").alias("staff_last_name")
)

# Iteratively build hierarchy paths
max_depth = 10
hierarchy = staff_hierarchy \
    .withColumn("path",
                F.when(F.col("staff_manager_key").isNotNull(),
                       F.array(F.col("staff_manager_key")))
                .otherwise(F.array())) \
    .withColumn("manager_keys",
                F.when(F.col("staff_manager_key").isNotNull(),
                       F.array(F.col("staff_manager_key")))
                .otherwise(F.array()))

for i in range(1, max_depth + 1):
    hierarchy = hierarchy.withColumn(
        "last_manager_key",
        F.when(F.size(F.col("manager_keys")) >= i, F.element_at(F.col("manager_keys"), i))
    )

    hierarchy = hierarchy.join(
        staff_hierarchy.select(
            F.col("staff_key").alias(f"mgr_key_{i}"),
            F.col("staff_manager_key").alias(f"next_mgr_key_{i}")
        ),
        F.col("last_manager_key") == F.col(f"mgr_key_{i}"),
        "left"
    ).withColumn(
        "new_manager_key",
        F.when(
            (F.col(f"next_mgr_key_{i}").isNotNull()) &
            (~F.array_contains(F.col("manager_keys"), F.col(f"next_mgr_key_{i}"))),
            F.col(f"next_mgr_key_{i}")
        )
    ).withColumn(
        "manager_keys",
        F.when(
            F.col("new_manager_key").isNotNull(),
            F.concat(F.col("manager_keys"), F.array(F.col("new_manager_key")))
        ).otherwise(F.col("manager_keys"))
    ).withColumn(
        "path",
        F.col("manager_keys")
    ).drop(f"mgr_key_{i}", f"next_mgr_key_{i}", "new_manager_key", "last_manager_key")

    new_additions = hierarchy.filter(F.size(F.col("manager_keys")) > i).count()
    if new_additions == 0:
        break

# Flatten hierarchy into bridge table
bridge_staff_hierarchy = hierarchy.select(
    F.col("staff_key"),
    F.col("staff_first_name"),
    F.col("staff_last_name"),
    F.posexplode(F.col("path")).alias("level_idx", "staff_manager_key")
).withColumn("level", F.col("level_idx") + 1) \
 .drop("level_idx") \
 .filter(F.col("staff_manager_key").isNotNull())

# Add manager names to bridge
bridge_staff_hierarchy = bridge_staff_hierarchy.join(
    staff_silver.select(
        xxhash64(F.col("staff_id")).alias("staff_manager_key"),
        F.col("first_name").alias("staff_manager_first_name"),
        F.col("last_name").alias("staff_manager_last_name")
    ),
    "staff_manager_key",
    "left"
)

write_gold_table(bridge_staff_hierarchy, "dim_staff_hierarchy")

# Build dim_staff with denormalized attributes
dim_staff = staff_silver \
    .join(address_with_location, staff_silver.address_id == address_with_location.address_id, "left") \
    .join(store_silver, staff_silver.store_id == store_silver.store_id, "left") \
    .join(
        bridge_staff_hierarchy.filter(col("level") == 1)
        .select("staff_key", "staff_manager_first_name", "staff_manager_last_name"),
        xxhash64(col("staff_id")) == bridge_staff_hierarchy["staff_key"],
        "left"
    ) \
    .withColumn("staff_key", xxhash64(col("staff_id"))) \
    .withColumn(
        "staff_address",
        F.when(
            col("address2").isNotNull(),
            F.concat_ws(", ", col("address"), col("address2"))
        ).otherwise(col("address"))
    ) \
    .select(
        "staff_key",
        "staff_id",
        store_silver.store_id,
        col("first_name").alias("staff_first_name"),
        col("last_name").alias("staff_last_name"),
        col("email").alias("staff_email"),
        "hired_date",
        "staff_address",
        address_with_location.city.alias("staff_city"),
        address_with_location.country.alias("staff_country"),

        "staff_manager_first_name",

        "staff_manager_last_name"

    )

write_gold_table(dim_staff, "dim_staff")
logger.info(f"GOLD: dim_staff completed in {time.time() - start_time:.2f}s")


In [0]:
# ==============================================================================
# GOLD: DIM_STORE (SCD TYPE 2 - Tracks Manager Changes)
# ==============================================================================
from pyspark.sql.functions import sha2, concat_ws, lit
from delta.tables import DeltaTable

logger.info("GOLD: Building dim_store with SCD Type 2")
start_time = time.time()

# Build store with denormalized address and location
store_with_address = store_silver.join(
    address_silver,
    store_silver.address_id == address_silver.address_id,
    "left"
).select(
    store_silver["*"],
    address_silver.address,
    address_silver.address2,
    address_silver.postal_code,
    address_silver.city_id
)

store_with_city = store_with_address.join(
    city_silver,
    store_with_address.city_id == city_silver.city_id,
    "left"
).select(
    store_with_address["*"],
    city_silver.city,
    city_silver.country_id
)

dim_store_raw = store_with_city.join(
    country_silver,
    store_with_city.country_id == country_silver.country_id,
    "left"
).select(
    store_with_city.store_id,
    store_with_city.store_manager_id,
    store_with_city.address_id,
    store_with_city.address,
    store_with_city.address2,
    city_silver.city.alias("city"),
    country_silver.country,
    store_with_city.postal_code,
    store_with_city.last_update
)

# Add manager names (denormalized for convenience)
dim_store_raw = dim_store_raw.join(
    staff_silver,
    dim_store_raw.store_manager_id == staff_silver.staff_id,
    "left"
).select(
    dim_store_raw["*"],
    staff_silver.staff_id.alias("manager_id"),  # Business key for manager
    staff_silver.first_name.alias("store_manager_first_name"),
    staff_silver.last_name.alias("store_manager_last_name"),
)

# Prepare new records with SCD2 columns
dim_store_new = dim_store_raw \
    .withColumn("store_key", xxhash64(col("store_id"))) \
    .withColumn(
        "store_address",
        F.when(
            col("address2").isNotNull(),
            F.concat_ws(", ", col("address"), col("address2"))
        ).otherwise(col("address"))
    ) \
    .withColumn("effective_date", current_timestamp()) \
    .withColumn("end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True)) \
    .select(
        "store_key",
        "store_id",
        "manager_id",
        "store_manager_first_name",
        "store_manager_last_name",
        "store_address",
        "city",
        "country",
        "postal_code",
        "last_update",
        "effective_date",
        "end_date",
        "is_current"
    )

# Check if table exists
table_exists = spark.catalog.tableExists("wheelie.data_warehouse.dim_store")

if not table_exists:
    # First load - just write all records
    logger.info("GOLD: dim_store first load (all records current)")
    write_gold_table(dim_store_new, "dim_store", mode="overwrite")
else:
    # SCD Type 2 MERGE logic (tracks manager changes only)
    logger.info("GOLD: dim_store incremental load with SCD Type 2")

    delta_table = DeltaTable.forName(spark, "wheelie.data_warehouse.dim_store")

    # Step 1: Close expired records where manager changed
    delta_table.alias("target").merge(
        dim_store_new.alias("source"),
        """target.store_id = source.store_id
           AND target.is_current = TRUE
           AND target.manager_id != source.manager_id"""
    ).whenMatchedUpdate(
        set={
            "is_current": lit(False),
            "end_date": current_timestamp()
        }
    ).execute()

    # Step 2: Insert new versions (for changed managers) and new stores
    # Get current state after closing expired records
    current_dim = spark.table("wheelie.data_warehouse.dim_store") \
        .filter(col("is_current") == True)

    # Find records that need new versions (manager changed or new store)
    records_to_insert = dim_store_new.alias("source").join(
        current_dim.alias("target"),
        (col("source.store_id") == col("target.store_id")) &
        (col("source.manager_id") == col("target.manager_id")),
        "left_anti"
    )

    # Append new versions
    if records_to_insert.count() > 0:
        logger.info(f"GOLD: Inserting {records_to_insert.count()} new/changed store records")
        write_gold_table(records_to_insert, "dim_store", mode="append")

    else:

        logger.info("GOLD: No manager changes detected, no new records to insert")
        logger.info(f"GOLD: dim_store (SCD Type 2) completed in {time.time() - start_time:.2f}s")


In [0]:
# ==============================================================================
# GOLD: DIM_CAR + DIM_EQUIPMENT + EQUIPMENT BRIDGES
# ==============================================================================
from pyspark.sql.functions import collect_set, concat_ws, array_sort

logger.info("GOLD: Building dim_car, dim_equipment, and equipment bridges")
start_time = time.time()

# Build dim_equipment first (all possible equipment types)
dim_equipment = equipment_silver \
    .withColumn("equipment_key", xxhash64(col("equipment_id"))) \
    .select(
        "equipment_key",
        "equipment_id",
        "name",
        "type",
        "version",
    )

write_gold_table(dim_equipment, "dim_equipment")

# Build equipment groups: unique combinations of equipment assigned to cars
equipment_groups = inventory_equipment_silver.groupBy("inventory_id") \
    .agg(array_sort(collect_set("equipment_id")).alias("equipments_array")) \
    .withColumn("equipments", concat_ws(",", col("equipments_array"))) \
    .withColumn("equipment_group_key", xxhash64(col("equipments"))) \
    .select("equipment_group_key", "equipments", "equipments_array") \
    .distinct()

# Normalize equipment groups: explode to individual equipment
equipment_groups_normalized = equipment_groups \
    .withColumn("equipment_id", F.explode(col("equipments_array"))) \
    .withColumn("equipment_key", xxhash64(col("equipment_id"))) \
    .select("equipment_group_key", "equipment_key") \
    .distinct()

write_gold_table(equipment_groups_normalized, "bridge_equipment_group_equipment")

# Bridge from car to equipment group
equipment_group_bridge = inventory_equipment_silver.groupBy("inventory_id") \
    .agg(array_sort(collect_set("equipment_id")).alias("equipments_array")) \
    .withColumn("car_key", xxhash64(col("inventory_id"))) \
    .withColumn("equipments", concat_ws(",", col("equipments_array"))) \
    .withColumn("equipment_group_key", xxhash64(col("equipments"))) \
    .select("car_key", "equipment_group_key") \
    .distinct()

write_gold_table(equipment_group_bridge, "bridge_car_equipment")

# Build dim_car
dim_car = inventory_silver.alias("inv").join(
    car_silver.alias("car"),
    col("inv.car_id") == col("car.car_id"),
    "left"
).withColumn(
    "car_key", xxhash64(col("inv.inventory_id"))
).select(
    col("car_key"),
    col("car.car_id"),
    col("inv.inventory_id"),
    col("car.producer"),
    col("car.model"),
    col("car.rental_rate"),
    col("inv.production_year"),
    col("inv.fuel_type"),
    col("inv.license_plates"),
    col("inv.purchase_price"),
    col("inv.sell_price"),
    col("inv.store_id"),
    col("inv.last_update"),
)

write_gold_table(dim_car, "dim_car")
logger.info(f"GOLD: dim_car and equipment tables completed in {time.time() - start_time:.2f}s")


In [0]:
# ==============================================================================
# GOLD: DIM_CUSTOMER
# ==============================================================================

logger.info("GOLD: Building dim_customer")
start_time = time.time()

# Build customer with denormalized address location
customer_with_address = customer_silver.join(
    address_with_location,
    "address_id",
    "left"
)

dim_customer = customer_with_address \
    .withColumn("customer_key", xxhash64(col("customer_id"))) \
    .withColumn(
        "customer_address",
        F.when(
            col("address2").isNotNull(),
            F.concat_ws(", ", col("address"), col("address2"))
        ).otherwise(col("address"))
    ) \
    .select(
        "customer_key",
        "customer_id",
        col("first_name").alias("customer_first_name"),
        col("last_name").alias("customer_last_name"),
        col("email").alias("customer_email"),
        "birth_date",
        "customer_address",
        col("city").alias("customer_city"),
        col("country").alias("customer_country"),
    )


write_gold_table(dim_customer, "dim_customer", partition_by=["customer_country"])

logger.info(f"GOLD: dim_customer completed in {time.time() - start_time:.2f}s")
logger.info("=" * 70)

logger.info("DIMENSION LOAD JOB COMPLETED")
logger.info("=" * 70)
